# 🎮 DQN auf Breakout — Lokale Version (Laptop/CPU)
**Mnih et al. 2015 — Paper-treue Reimplementierung**

---

Diese Version ist **für deinen Laptop optimiert**:
- Kleinerer Replay Buffer (weniger RAM)
- Weniger Frames (realistisches Ziel für CPU)
- Automatisches Speichern damit nichts verloren geht
- Kein Live-Plot-Overhead

**Erwartung auf CPU:** ~20.000–80.000 Steps/Stunde je nach Prozessor

---

### Setup (einmalig im Terminal):
```bash
pip install jupyter notebook
pip install torch torchvision
pip install gymnasium[atari] ale-py
pip install autorom[accept-rom-license]
AutoROM --accept-license
```
Dann:
```bash
jupyter notebook
```

In [2]:
# ── ZELLE 1: System prüfen ───────────────────────────────────────
import torch
import platform
import psutil

ram_gb   = psutil.virtual_memory().total / 1e9
ram_free = psutil.virtual_memory().available / 1e9
device   = torch.device(
    'cuda' if torch.cuda.is_available() else
    'mps'  if torch.backends.mps.is_available() else 'cpu'
)

print('=== System-Info ===')
print(f'  OS:       {platform.system()} {platform.release()}')
print(f'  Python:   {platform.python_version()}')
print(f'  PyTorch:  {torch.__version__}')
print(f'  Device:   {device}')
print(f'  RAM:      {ram_free:.1f} GB frei / {ram_gb:.1f} GB gesamt')
print()

if device.type == 'cuda':
    print(f'  GPU:      {torch.cuda.get_device_name(0)}')
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'  VRAM:     {vram:.1f} GB')
    print('  ✅ GPU gefunden — Training wird deutlich schneller')
elif device.type == 'mps':
    print('  ✅ Apple Silicon (MPS) — gut für Training')
else:
    print('  ⚠️  Nur CPU — Training läuft, aber langsam (~1-3 Tage für gute Ergebnisse)')
    print('     Tipp: Notebook über Nacht laufen lassen, Checkpoints sichern')

# RAM-Warnung
if ram_free < 4.0:
    print(f'  ⚠️  Wenig freier RAM ({ram_free:.1f} GB) — Buffer wird klein gehalten')

=== System-Info ===
  OS:       Windows 10
  Python:   3.11.9
  PyTorch:  2.10.0+cpu
  Device:   cpu
  RAM:      12.6 GB frei / 33.8 GB gesamt

  ⚠️  Nur CPU — Training läuft, aber langsam (~1-3 Tage für gute Ergebnisse)
     Tipp: Notebook über Nacht laufen lassen, Checkpoints sichern


In [3]:
# ── ZELLE 2: Imports ─────────────────────────────────────────────
import random
import time
import os
from collections import deque, namedtuple
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import gymnasium as gym
import ale_py
gym.register_envs(ale_py)
from gymnasium.wrappers import (
    AtariPreprocessing,
    FrameStackObservation,
    RecordEpisodeStatistics,
)
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 100

print('✅ Alle Imports erfolgreich')

✅ Alle Imports erfolgreich


In [4]:
# ── ZELLE 3: Hyperparameter (RAM-adaptiv) ────────────────────────
from pathlib import Path
import psutil
ram_free_gb = psutil.virtual_memory().available / 1e9

# Replay Buffer automatisch an verfügbaren RAM anpassen
if ram_free_gb > 12:
    REPLAY_CAPACITY = 100_000
elif ram_free_gb > 6:
    REPLAY_CAPACITY = 50_000
else:
    REPLAY_CAPACITY = 25_000

# ── Paper-Werte (Table 1) ─────────────────────────────────────────
BATCH_SIZE           = 32
GAMMA                = 0.99
LR                   = 0.00025
RMSPROP_ALPHA        = 0.95
RMSPROP_EPS          = 0.01
EPSILON_START        = 1.0
EPSILON_END          = 0.1
EPSILON_DECAY_FRAMES = 500_000   # Paper: 1M — angepasst für CPU
REPLAY_START         = min(10_000, REPLAY_CAPACITY // 5)
TARGET_UPDATE_C      = 10_000
MAX_FRAMES           = 2_000_000 # Realistisches Ziel für Laptop
SAVE_DIR             = Path('dqn_checkpoints')
SAVE_DIR.mkdir(exist_ok=True)

device = torch.device(
    'cuda' if torch.cuda.is_available() else
    'mps'  if torch.backends.mps.is_available() else 'cpu'
)

print(f'Hyperparameter gesetzt:')
print(f'  Replay Buffer: {REPLAY_CAPACITY:,} Transitions (~{REPLAY_CAPACITY*4*84*84/1e6:.0f} MB RAM)')
print(f'  Replay Start:  {REPLAY_START:,}')
print(f'  Max Frames:    {MAX_FRAMES:,}')
print(f'  Device:        {device}')
print(f'  Checkpoints:   ./{SAVE_DIR}/')

Hyperparameter gesetzt:
  Replay Buffer: 100,000 Transitions (~2822 MB RAM)
  Replay Start:  10,000
  Max Frames:    2,000,000
  Device:        cpu
  Checkpoints:   ./dqn_checkpoints/


In [5]:
# ── ZELLE 4: Netz + Buffer + Hilfsfunktionen ─────────────────────

class DQN(nn.Module):
    """Nature 2015, Table 1 — exakte Architektur."""
    def __init__(self, n_actions):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(4, 32, kernel_size=8, stride=4), nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2), nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=1), nn.ReLU(),
        )
        self.fc = nn.Sequential(
            nn.Linear(64 * 7 * 7, 512), nn.ReLU(),
            nn.Linear(512, n_actions),
        )
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.Linear)):
                nn.init.kaiming_uniform_(m.weight, nonlinearity='relu')
                nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.fc(self.conv(x).flatten(1))


Transition = namedtuple('Transition', ['state','action','reward','next_state','done'])

class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)

    def push(self, s, a, r, ns, d):
        self.buffer.append(Transition(
            s.astype(np.uint8), a, r,
            ns.astype(np.uint8), d
        ))

    def sample(self, n):
        batch = random.sample(self.buffer, n)
        s  = torch.from_numpy(np.stack([t.state      for t in batch])).float().div(255).to(device)
        ns = torch.from_numpy(np.stack([t.next_state for t in batch])).float().div(255).to(device)
        a  = torch.tensor([t.action for t in batch], dtype=torch.long,  device=device)
        r  = torch.tensor([t.reward for t in batch], dtype=torch.float, device=device)
        d  = torch.tensor([t.done   for t in batch], dtype=torch.float, device=device)
        return s, a, r, ns, d

    def __len__(self): return len(self.buffer)


def epsilon(frame):
    frac = min(1.0, frame / EPSILON_DECAY_FRAMES)
    return EPSILON_START + frac * (EPSILON_END - EPSILON_START)


def make_env(seed=42):
    env = gym.make('ALE/Breakout-v5', frameskip=1)
    env = AtariPreprocessing(
        env, noop_max=30, frame_skip=4, screen_size=84,
        terminal_on_life_loss=True, grayscale_obs=True, scale_obs=False
    )
    env = FrameStackObservation(env, stack_size=4)
    env = RecordEpisodeStatistics(env)
    env.reset(seed=seed)
    return env


def save_checkpoint(frame, policy_net, target_net, optimizer, ep_rewards, all_rewards, all_losses, all_qvals):
    path = SAVE_DIR / f'dqn_breakout_frame{frame}.pt'
    torch.save({
        'frame':        frame,
        'policy_state': policy_net.state_dict(),
        'target_state': target_net.state_dict(),
        'optimizer':    optimizer.state_dict(),
        'ep_rewards':   list(ep_rewards),
        'all_rewards':  all_rewards,
        'all_losses':   all_losses,
        'all_qvals':    all_qvals,
    }, path)
    return path
def load_checkpoint(path, policy_net, target_net, optimizer):
    ckpt = torch.load(path, map_location=device, weights_only=False)
    policy_net.load_state_dict(ckpt['policy_state'])
    target_net.load_state_dict(ckpt['target_state'])
    optimizer.load_state_dict(ckpt['optimizer'])
    # Lädt Daten sicher, auch aus alten/fehlerhaften Checkpoints
    frames = ckpt.get('frame', 0)
    epr    = deque(ckpt.get('ep_rewards', ckpt.get('rewards', [])), maxlen=100)
    ar     = ckpt.get('all_rewards', ckpt.get('losses', []))
    al     = ckpt.get('all_losses',  ckpt.get('qvals', []))
    aq     = ckpt.get('all_qvals', [])
    return frames, epr, ar, al, aq


✅ Alle Klassen und Funktionen definiert


In [6]:
# ── ZELLE 5: Sanity-Check ────────────────────────────────────────
print('Sanity-Check läuft...\n')

# 1. Netz-Shape
net = DQN(4).to(device)
x   = torch.randn(8, 4, 84, 84, device=device)
out = net(x)
assert out.shape == (8, 4), f'Falsche Shape: {out.shape}'
params = sum(p.numel() for p in net.parameters())
print(f'  ✅ Netz-Shape: {tuple(x.shape)} → {tuple(out.shape)}')
print(f'     Parameter: {params:,} ({params/1e6:.2f}M)')

# 2. Buffer
buf = ReplayBuffer(500)
for _ in range(200):
    buf.push(
        np.random.randint(0,255,(4,84,84),dtype=np.uint8),
        random.randint(0,3), random.random(),
        np.random.randint(0,255,(4,84,84),dtype=np.uint8), False
    )
s,a,r,ns,d = buf.sample(32)
assert s.max() <= 1.0
assert s.shape == (32,4,84,84)
print(f'\n  ✅ Buffer: 200 Transitions, states normiert auf [0,1]')

# 3. Train Step + Gradient-Isolation
tnet = DQN(4).to(device)
tnet.load_state_dict(net.state_dict())
opt  = optim.RMSprop(net.parameters(), lr=LR, alpha=RMSPROP_ALPHA, eps=RMSPROP_EPS)

with torch.no_grad():
    q_t = tnet(ns).max(1).values
q_p  = net(s).gather(1, a.unsqueeze(1)).squeeze()
loss = F.smooth_l1_loss(q_p, r + GAMMA * q_t * (1-d))
loss.backward()

for p in tnet.parameters():
    assert p.grad is None, '❌ Target-Netz hat Gradient — Bug!'

print(f'\n  ✅ Train-Step: loss={loss.item():.4f}')
print(f'  ✅ Gradient-Isolation: θ⁻.grad = None (korrekt)')

# 4. Epsilon Schedule
eps_vals = [epsilon(f) for f in [0, 250_000, 500_000, 1_000_000]]
print(f'\n  ✅ ε-Schedule: {[f"{e:.2f}" for e in eps_vals]} (Start→Ende)')

print('\n' + '='*45)
print('  Alle Checks bestanden ✅ — Training kann starten!')
print('='*45)

Sanity-Check läuft...

  ✅ Netz-Shape: (8, 4, 84, 84) → (8, 4)
     Parameter: 1,686,180 (1.69M)

  ✅ Buffer: 200 Transitions, states normiert auf [0,1]

  ✅ Train-Step: loss=1.0044
  ✅ Gradient-Isolation: θ⁻.grad = None (korrekt)

  ✅ ε-Schedule: ['1.00', '0.55', '0.10', '0.10'] (Start→Ende)

  Alle Checks bestanden ✅ — Training kann starten!


In [8]:
# ── ZELLE 6: Training initialisieren ─────────────────────────────
# Hier kannst du einen Checkpoint laden falls du weitermachen willst:
CHECKPOINT_TO_LOAD = None  # z.B. 'dqn_checkpoints/dqn_breakout_frame500000.pt'

env        = make_env()
n_actions  = env.action_space.n
policy_net = DQN(n_actions).to(device)
target_net = DQN(n_actions).to(device)
target_net.load_state_dict(policy_net.state_dict())
target_net.eval()
optimizer  = optim.RMSprop(policy_net.parameters(),
                            lr=LR, alpha=RMSPROP_ALPHA, eps=RMSPROP_EPS)
memory     = ReplayBuffer(REPLAY_CAPACITY)

# Tracking-Listen
ep_rewards  = deque(maxlen=100)
all_rewards = []
all_losses  = []
all_qvals   = []
losses_buf  = deque(maxlen=1000)
qvals_buf   = deque(maxlen=1000)
frame_count = 0
steps_done  = 0
episode     = 0

# Checkpoint laden?
if CHECKPOINT_TO_LOAD and Path(CHECKPOINT_TO_LOAD).exists():
    frame_count, ep_rewards, all_rewards, all_losses, all_qvals = load_checkpoint(
        CHECKPOINT_TO_LOAD, policy_net, target_net, optimizer
    )
    print(f'✅ Checkpoint geladen: Frame {frame_count:,}')
else:
    print('Frisch gestartet (kein Checkpoint)')

# Buffer vorfüllen
print(f'\nBuffer wird vorgefüllt ({REPLAY_START:,} zufällige Steps)...')
obs, _ = env.reset()
for i in range(REPLAY_START):
    a = env.action_space.sample()
    no, r, term, trunc, _ = env.step(a)
    done = term or trunc
    memory.push(np.array(obs), a, np.clip(r,-1,1), np.array(no), done)
    obs = no if not done else env.reset()[0]
    if (i+1) % 2000 == 0:
        print(f'  {i+1:,} / {REPLAY_START:,}...')

obs, _ = env.reset()
print(f'\n✅ Buffer: {len(memory):,} Transitions')
print('Bereit. Nächste Zelle ausführen um Training zu starten.')

NamespaceNotFound: Namespace ALE not found. Have you installed the proper package for ALE?

In [ ]:
# ── ZELLE 7: Training ────────────────────────────────────────────
# Kann jederzeit mit dem Stop-Button (■) unterbrochen werden.
# Checkpoint wird alle 100k Frames automatisch gespeichert.

ep_reward  = 0.0
t_start    = time.time()
log_every  = 5_000    # Textausgabe alle N Frames
plot_every = 50_000   # Plot alle N Frames
save_every = 100_000  # Checkpoint alle N Frames

print(f'Training startet bei Frame {frame_count:,}')
print(f'Ziel: {MAX_FRAMES:,} Frames')
print(f'Checkpoint alle {save_every:,} Frames → ./{SAVE_DIR}/')
print('Stoppen: Kernel → Interrupt (■ oben)')
print('─'*55)

try:
    while frame_count < MAX_FRAMES:

        # ε-greedy Action
        eps = epsilon(frame_count)
        if random.random() < eps:
            action = env.action_space.sample()
        else:
            with torch.no_grad():
                s_t    = torch.from_numpy(
                    np.array(obs).astype(np.float32) / 255.0
                ).unsqueeze(0).to(device)
                q      = policy_net(s_t)
                action = q.argmax(1).item()
                qvals_buf.append(q.max().item())

        # Environment Step
        next_obs, reward, terminated, truncated, _ = env.step(action)
        done      = terminated or truncated
        memory.push(np.array(obs), action, np.clip(reward,-1,1),
                    np.array(next_obs), done)
        obs        = next_obs
        ep_reward += reward
        frame_count += 1

        # ── Algorithm 1 inner loop ───────────────────────────────
        s, a, r, ns, d = memory.sample(BATCH_SIZE)

        q_pred = policy_net(s).gather(1, a.unsqueeze(1)).squeeze()

        with torch.no_grad():
            q_next   = target_net(ns).max(1).values
            q_target = r + GAMMA * q_next * (1.0 - d)

        loss = F.smooth_l1_loss(q_pred, q_target)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(policy_net.parameters(), 10.0)
        optimizer.step()

        losses_buf.append(loss.item())
        steps_done += 1

        # Target Update
        if steps_done % TARGET_UPDATE_C == 0:
            target_net.load_state_dict(policy_net.state_dict())

        # Episode Ende
        if done:
            episode  += 1
            ep_rewards.append(ep_reward)
            ep_reward = 0.0
            obs, _    = env.reset()

        # ── Log ─────────────────────────────────────────────────
        if frame_count % log_every == 0:
            elapsed  = time.time() - t_start
            fps      = frame_count / elapsed
            eta_h    = (MAX_FRAMES - frame_count) / fps / 3600
            mean_r   = np.mean(ep_rewards) if ep_rewards else 0
            mean_l   = np.mean(losses_buf) if losses_buf else 0
            mean_q   = np.mean(qvals_buf)  if qvals_buf  else 0
            pct      = frame_count / MAX_FRAMES * 100

            all_rewards.append(mean_r)
            all_losses.append(mean_l)
            all_qvals.append(mean_q)

            print(
                f'  [{pct:5.1f}%] Frame {frame_count:>7,} | '
                f'Ep {episode:>4} | ε={eps:.3f} | '
                f'R100={mean_r:>6.2f} | Q={mean_q:>5.3f} | '
                f'L={mean_l:.4f} | {fps:.0f} fps | ETA {eta_h:.1f}h'
            )

        # ── Plot ────────────────────────────────────────────────
        if frame_count % plot_every == 0 and len(all_rewards) > 1:
            fig, axes = plt.subplots(1, 3, figsize=(14, 3))
            fig.suptitle(
                f'DQN Breakout  —  Frame {frame_count:,}  |  '
                f'Episode {episode}  |  ε={eps:.3f}',
                fontsize=11
            )
            x_axis = [i * log_every for i in range(len(all_rewards))]

            axes[0].plot(x_axis, all_rewards, '#2ecc71', linewidth=1.5)
            axes[0].set_title('Ø Reward (100 Ep)');  axes[0].set_xlabel('Frames')
            axes[0].axhline(0, color='gray', linewidth=0.5, ls='--')
            axes[0].grid(True, alpha=0.3)

            axes[1].plot(x_axis, all_losses, '#e74c3c', linewidth=1.5)
            axes[1].set_title('Loss (Huber)');  axes[1].set_xlabel('Frames')
            axes[1].grid(True, alpha=0.3)

            if len(all_qvals) == len(x_axis):
                if len(all_qvals) == len(x_axis): axes[2].plot(x_axis, all_qvals, '#3498db', linewidth=1.5)
            else:
                axes[2].text(0.5, 0.5, 'Historie fehlt', ha='center')
            axes[2].set_title('Ø Q-Value');  axes[2].set_xlabel('Frames')
            axes[2].grid(True, alpha=0.3)

            plt.tight_layout()
            plt.savefig(SAVE_DIR / f'plot_frame{frame_count}.png',
                        dpi=80, bbox_inches='tight')
            plt.show()

        # ── Checkpoint ──────────────────────────────────────────
        if frame_count % save_every == 0:
            path = save_checkpoint(
                frame_count, policy_net, target_net,
                optimizer, ep_rewards, all_rewards, all_losses, all_qvals, all_qvals
            )
            print(f'  💾 Gespeichert: {path}')

except KeyboardInterrupt:
    print('\n⏸  Training unterbrochen.')
    path = save_checkpoint(
        frame_count, policy_net, target_net,
        optimizer, ep_rewards, all_rewards, all_losses, all_qvals, all_qvals
    )
    print(f'💾 Letzter Stand gespeichert: {path}')
    print(f'   Weitermachen: CHECKPOINT_TO_LOAD = "{path}" in Zelle 6 setzen')

finally:
    env.close()

print('\nTraining abgeschlossen.')


In [ ]:
# ── ZELLE 8: Ergebnis-Plot (nach Training oder aus Checkpoint) ────
if len(all_rewards) > 1:
    x_axis = [i * log_every for i in range(len(all_rewards))]

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle(f'DQN Breakout — Trainingsverlauf ({frame_count:,} Frames)',
                 fontsize=13, fontweight='bold')

    axes[0].plot(x_axis, all_rewards, '#2ecc71', linewidth=2)
    axes[0].fill_between(x_axis, all_rewards, alpha=0.15, color='#2ecc71')
    axes[0].set_title('Ø Reward (letzte 100 Episoden)', fontsize=11)
    axes[0].set_xlabel('Frames');  axes[0].set_ylabel('Reward')
    axes[0].axhline(0, color='gray', linewidth=0.5, ls='--')
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(x_axis, all_losses, '#e74c3c', linewidth=2)
    axes[1].set_title('Training Loss (Huber)', fontsize=11)
    axes[1].set_xlabel('Frames');  axes[1].set_ylabel('Loss')
    axes[1].grid(True, alpha=0.3)

    if len(all_qvals) == len(x_axis):
        if len(all_qvals) == len(x_axis): axes[2].plot(x_axis, all_qvals, '#3498db', linewidth=2)
    else:
        axes[2].text(0.5, 0.5, 'Historie fehlt', ha='center')
    axes[2].fill_between(x_axis, all_qvals, alpha=0.15, color='#3498db')
    axes[2].set_title('Ø Q-Value (Exploitation)', fontsize=11)
    axes[2].set_xlabel('Frames');  axes[2].set_ylabel('Q')
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(SAVE_DIR / 'final_plot.png', dpi=100, bbox_inches='tight')
    plt.show()
    print(f'Plot gespeichert: {SAVE_DIR}/final_plot.png')

    print(f'\n=== Zusammenfassung ===')
    print(f'  Frames trainiert:  {frame_count:,}')
    print(f'  Episoden:          {episode}')
    print(f'  Bester Ø Reward:   {max(all_rewards):.2f}')
    print(f'  Letzter Ø Reward:  {all_rewards[-1]:.2f}')
    print(f'  Checkpoints in:    ./{SAVE_DIR}/')
else:
    print('Noch keine Daten — Training erst starten (Zelle 7)')


In [ ]:
# ── ZELLE 9: Was bedeuten die Kurven? ────────────────────────────
print('=== INTERPRETATIONSHILFE ===')
print()
print('📈 REWARD-KURVE (grün) — das Wichtigste:')
print('   Steigt sie? → Agent lernt')
print('   Bleibt sie bei 0-1? → noch zu früh, braucht mehr Frames')
print()
print('   Typischer Verlauf auf CPU:')
print('   0 – 200k Frames:   Reward ~0-2   (fast zufällig)')
print('   200k – 500k:       Reward ~2-10  (erste Paddle-Kontrolle)')
print('   500k – 1M:         Reward ~5-20  (gezieltes Spielen)')
print('   1M – 2M:           Reward ~15-40 (erkennbare Strategie)')
print()
print('📉 LOSS-KURVE (rot):')
print('   Steigt zuerst → normal, Netz lernt überhaupt etwas')
print('   Stabilisiert sich → gut')
print('   Explodiert (>10) → Problem, meld dich!')
print()
print('📊 Q-VALUE-KURVE (blau):')
print('   Sollte langsam und stetig steigen')
print('   Explodiert (>100) → Target-Network-Bug')
print()
print('=== CHECKPOINT WEITERMACHEN ===')
print('In Zelle 6:')
print('  CHECKPOINT_TO_LOAD = "dqn_checkpoints/dqn_breakout_frame100000.pt"')
print('Dann Zellen 6 und 7 neu ausführen.')